Code to generate select DHAD sequences from unique organisms (labeled by UniProt ID) filtered by KARI distance. 

The select distance is represented by N in the below section (i.e. N=2 means that KARI must be within 2 genes of the DHAD in the Genomic Neighborhood Network (GNN)):
"This section is to filter genomes based on N (defined number of genes upstream/downstream of DHAD)"

DHAD and KARI locations in the Genomic Distance is provided by Enzyme Function Initiative (EFI) database. (Zallot et al., 2019)


In [1]:
import os, fnmatch
import re
import pandas as pd
from io import StringIO
import sys
import requests

def get_species_fasta_from_uniprot_id(uniprot_id):

    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}"

    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        #print(data)
        organism = data["organism"]["scientificName"]
        AAsequence = data["sequence"]["value"]
        #print(organism)
        #print(AAsequence)
        return organism, AAsequence
    
    else:
        print(f"Failed to retrieve data for UniProt ID: {uniprot_id}, status code: {response.status_code}")

def create_grouped_dict_from_df(df, key_col, value_col):
    """
    Creates a dictionary from a Pandas DataFrame using specified columns for keys and values,
    and groups similar keys.

    Args:
      df: The Pandas DataFrame.
      key_col: The column name to use for dictionary keys.
      value_col: The column name to use for dictionary values.

    Returns:
      A dictionary with keys from key_col and values as lists of grouped values from value_col.
    """
    if key_col not in df.columns or value_col not in df.columns:
        raise ValueError("Specified column names not found in DataFrame.")
    
    grouped_dict = df.groupby(key_col)[value_col].apply(list).to_dict()
    return grouped_dict


def search_forFunctionValue_inGNNdf(values, values_to_search: list):
    # returns a dataframe with rows containing values that match any of the values in values_to_search
    valuepattern = '|'.join(values_to_search)
    found = values[values['Function'].str.contains(valuepattern, case=False, na=False)]
    notfound = values[~values['Function'].str.contains(valuepattern, case=False, na=False)]
    #print(values)
    return found

def search_forIDValue_inGNNdf(values, values_to_search: list):
    # returns a dataframe with rows containing values that match any of the values in values_to_search
    valuepattern = '|'.join(values_to_search)
    found = values[values['ID'].str.contains(valuepattern, case=False, na=False)]
    notfound = values[~values['ID'].str.contains(valuepattern, case=False, na=False)]
    #print(values)
    return found


def remove_organisms_NotHavingValues(df, column_name, values_list):
    """
    Removes rows from a DataFrame where the specified column's values
    are not present in the given list.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to check.
        values_list (list): A list of values to keep. Rows with values
                           not in this list will be removed.

    Returns:
        pd.DataFrame: A new DataFrame with the filtered rows.
    """

    filtered_df = df[df[column_name].isin(values_list)]
    return filtered_df


def keep_rows_around_candidate(df, candidate_list, n):
    """
    Keeps N rows above and below the row that contains any value from the candidate list in each group by 'Genome'.

    Args:
        df (pd.DataFrame): The input DataFrame.
        candidate_list (list): List of candidate values to search for.
        n (int): Number of rows to keep above and below the found row.

    Returns:
        pd.DataFrame: A new DataFrame with the specified rows kept.
    """
    if n > 101:
        raise ValueError("N should not exceed 101, unless you have a bigger geneMAP (greater than 100 genes upstream/downstream).")
    indices_to_keep = []
    for candidate in candidate_list:
        candidate_indices = df[df['ID'] == candidate].index.tolist()
        for idx in candidate_indices:
            indices_to_keep.extend(range(max(idx - n, 0), min(idx + n + 1, len(df))))
    return df.loc[indices_to_keep].reset_index(drop=True)


In [ ]:
home_path = '/Users/jeremydcortez/Bioconda_Playground/25-0214/001'

os.chdir(home_path)

file_path = 'filtered_GNNgeneGraph0214_001.csv'

## candidateIDs_path represents the .csv file
candidateIDs_path = '/Users/jeremydcortez/Bioconda_Playground/25-0214/001/filtered_UNIPROT_IDDHAD0214_002.csv'

In [57]:
GNNgeneMap_df= pd.read_csv(file_path) #change file!
#print(GNNgeneGraph_df)
UniprotDHAD_candidates_df = pd.read_csv(candidateIDs_path) #change file!
UniprotDHAD_allDHADCandidates = UniprotDHAD_candidates_df['ID'].tolist()
#print(f'There are {len(UniprotDHAD_allDHADCandidates)} DHAD candidates in the dataset')
#print(UniprotDHAD_allDHADCandidates)


groupedbyGenome_GNNgeneGraph = create_grouped_dict_from_df(GNNgeneMap_df, 'Genome', 'Function')
species = list(groupedbyGenome_GNNgeneGraph.keys())
#print(groupedbyGenome_GNNgeneGraph)


In [58]:
# Remove duplicate rows based on the 'ID' column
GNNgeneMap_df_unique = GNNgeneMap_df.drop_duplicates(subset='ID')

# Save the resulting DataFrame to a CSV file
GNNgeneMap_df_unique.to_csv('filtered_GNNgeneGraph0214_002.csv', index=False)
print('CSV file with unique IDs generated successfully.')

CSV file with unique IDs generated successfully.


In [59]:
os.chdir('/Users/jeremydcortez/Bioconda_Playground/25-0214/001')
print(f'path changed to {os.getcwd()}')
# change input file for GNNgeneMap to 002 version
file_path = '/Users/jeremydcortez/Bioconda_Playground/25-0214/001/filtered_GNNgeneGraph0214_002.csv'
print(f'file input changed to {file_path}')

# input new csv into df (rewriting the original df)
GNNgeneMap_df= pd.read_csv(file_path) #DO NOT change this file!


path changed to /Users/jeremydcortez/Bioconda_Playground/25-0214/001
file input changed to /Users/jeremydcortez/Bioconda_Playground/25-0214/001/filtered_GNNgeneGraph0214_002.csv


In [60]:
# Find the IDs in UniprotDHAD_allDHADCandidates that are not in GNNgeneGraph_df
missing_DHADs = [id for id in UniprotDHAD_allDHADCandidates if id not in GNNgeneMap_df['ID'].values]

if len(missing_DHADs) > 0:
    print(f'Total missing DHADs from the mapped list: {len(missing_DHADs)}')
    print('remap the following DHADs:')
    for k in missing_DHADs:
        print(k)
else:
    print('All DHADs are present in the GNN mapped list')

All DHADs are present in the GNN mapped list


In [61]:
# group the dataframe by 'Genome'
groupedGNNmap_byGenome = GNNgeneMap_df.groupby('Genome')

# Create a dictionary to hold the separate dataframes
genome_dfs = {}

# Group the dataframe by 'Genome' and create separate dataframes
for genome, group in groupedGNNmap_byGenome:
    genome_dfs[genome] = group.reset_index(drop=True)

# Example: Accessing a specific dataframe
# print(genome_dfs['Acaryochloris marina (strain MBIC 11017)'])

In [62]:
keys_with_more_than_205_values = [key for key, df in genome_dfs.items() if len(df) > 205]
print(f'There are {len(keys_with_more_than_205_values)} genomes with more than 205 genes (likely each organism has multiple DHADs)')
#print(keys_with_more_than_205_values)

There are 14 genomes with more than 205 genes (likely each organism has multiple DHADs)


--------- This section is to filter genomes based on N (defined number of genes upstream/downstream of DHAD) --------

In [74]:

# Change N, the number of genes to keep upstream and downstream of DHAD candidates



N = 2  # parameter to change!



# Apply the function to each dataframe in genome_dfs
filtered_genome_dfs = {genome: keep_rows_around_candidate(df, UniprotDHAD_allDHADCandidates, N) for genome, df in genome_dfs.items()}

# Example: Accessing a specific filtered dataframe
# print(filtered_genome_dfs['Acaryochloris marina (strain MBIC 11017)'])
# change directory to save the new csv file
os.makedirs('GNN'+str(N), exist_ok=False)
os.chdir('GNN'+str(N))

# Save the filtered dataframes to separate CSV files
combined_df = pd.concat(filtered_genome_dfs.values(), ignore_index=True)
rowsdeletedfile = 'filtered_GNNgeneGraph0214_003_GNN'+str(N)+'.csv'
combined_df.to_csv(rowsdeletedfile, index=False)
print('CSV file generated successfully.')


CSV file generated successfully.


In [75]:

# change input file for GNNgeneMap to 003 version
file_path = str(rowsdeletedfile)

GNNgeneMap_df= pd.read_csv(file_path) #DO NOT change file!

# Find the IDs in UniprotDHAD_allDHADCandidates that are not in GNNgeneGraph_df
missing_DHADs = [id for id in UniprotDHAD_allDHADCandidates if id not in GNNgeneMap_df['ID'].values]

if len(missing_DHADs) > 0:
    print(f'Total missing DHADs from the mapped list: {len(missing_DHADs)}')
    print('remap the following DHADs:')
    for k in missing_DHADs:
        print(k)
else:
    print('All DHADs are present in the GNN mapped list.\n')



# search GNNgeneGraph_df for DHAD that exist in UniprotDHAD_allDHADCandidates
foundDHAD = search_forIDValue_inGNNdf(GNNgeneMap_df, UniprotDHAD_allDHADCandidates)
# group found organisms with DHAD from list
groupedbyGenome_foundDHAD = create_grouped_dict_from_df(foundDHAD, 'Genome', 'Function')
speciesDHAD = list(groupedbyGenome_foundDHAD.keys())


print(f'Found {len(foundDHAD)} DHADs')
print(f'Found {len(speciesDHAD)} organisms with DHADs')


All DHADs are present in the GNN mapped list.

Found 401 DHADs
Found 381 organisms with DHADs


In [76]:
#find any KARI
# old KARI search with 'IPR008927', 'IPR014359', 'IPR013328', 'IPR013023', 'IPR013116'
# IPR008927 = C terminal ketol-acid reductoisomerase *also labeled as phosphogluconate dehydrogenase
# IPR014359 = Ketol-acid reductoisomerase, prokaryotic
# IPR013023 = Ketol-acid reductoisomerase, NADP-dependent
# IPR013116 = Ketol-acid reductoisomerase, N-terminal *may not be necessary

KARIIDs = ['IPR014359', 'IPR013023']

foundKARI = search_forFunctionValue_inGNNdf(GNNgeneMap_df, KARIIDs)
groupedbyGenome_foundKARI = create_grouped_dict_from_df(foundKARI, 'Genome', 'Function')
speciesKARI = list(groupedbyGenome_foundKARI.keys())
groupedbyUniprotID_foundKARI = create_grouped_dict_from_df(foundKARI, 'Genome', 'ID')
UniprotFoundKARIID = list(groupedbyUniprotID_foundKARI.items()) ## KARIs found listed here!

print(f'Found {len(foundKARI)} KARIs in {len(speciesKARI)} different organisms when searched with {KARIIDs}.')

#print(foundKARI)


Found 25 KARIs in 25 different organisms when searched with ['IPR014359', 'IPR013023'].


In [78]:
# remove organisms that do not have DHADs and KARIs
# 

GNNgeneGraph_df_filtered_FINAL = remove_organisms_NotHavingValues(GNNgeneMap_df, 'Genome', speciesKARI)
# search GNNgeneGraph_df for DHAD that exist in UniprotDHAD_allDHADCandidates
foundDHADwKARIfinal = search_forIDValue_inGNNdf(GNNgeneGraph_df_filtered_FINAL, UniprotDHAD_allDHADCandidates)

# group found organisms with DHAD from list
groupedbyGenome_foundDHADfinal = create_grouped_dict_from_df(foundDHADwKARIfinal, 'Genome', 'Function')
speciesDHADFINAL = list(groupedbyGenome_foundDHADfinal.keys())
groupedbyUniprotID_foundDHADfinal = create_grouped_dict_from_df(foundDHADwKARIfinal, 'Genome', 'ID')
UniprotFoundDHADIDfinal = list(groupedbyUniprotID_foundDHADfinal.items())

foundKARIfinal = search_forFunctionValue_inGNNdf(GNNgeneGraph_df_filtered_FINAL, KARIIDs)
groupedbyGenome_foundKARIfinal = create_grouped_dict_from_df(foundKARIfinal, 'Genome', 'Function')
speciesKARIFINAL = list(groupedbyGenome_foundKARIfinal.keys())

groupedbyUniprotID_foundKARIfinal = create_grouped_dict_from_df(foundKARIfinal, 'Genome', 'ID')
UniprotFoundKARIIDfinal = list(groupedbyUniprotID_foundKARIfinal.items())

print(f'Found {len(foundKARIfinal)} KARIs in {len(speciesKARIFINAL)} different organisms.')
print(f'Found {len(foundDHADwKARIfinal)} DHADs with KARI in {len(speciesDHADFINAL)} different organisms.')

#print(UniprotFoundDHADIDfinal)

for i in UniprotFoundDHADIDfinal:
    for j in UniprotFoundKARIIDfinal:
        if i[0] == j[0]:
            #print(f'Found {i[0]} with DHAD and KARI')
            #print(f'KARI: {j[1]}')
            #print(f'DHAD: {i[1]}')
            #print('')
            if len(i[1]) > 1:
                print('')
                print(f'Multiple DHADs found in {i[0]}')
                print(f'DHAD: {i[1]}')
                print('')


Found 25 KARIs in 25 different organisms.
Found 25 DHADs with KARI in 25 different organisms.


In [ ]:


DHADtoanalysis = []
for i in UniprotFoundDHADIDfinal:
    if len(i[1]) > 1:
        for j in i[1]:
            DHADtoanalysis.append(j)
    else:
        DHADtoanalysis.append(i[1][0])
        #print(i[1][0])

print(DHADtoanalysis)
print(f'There are {len(DHADtoanalysis)} DHADs to analyze.')



['Q81S26', 'Q9XBI3', 'A9VPN7', 'A0RCL3', 'Q7VRL8', 'Q491Z0', 'Q056W3', 'B2TIR2', 'A0Q0E8', 'U5MKI6', 'C4ZZ41', 'B5YY20', 'B9EBF6', 'E0THN6', 'E1R5W9', 'A6QIQ0', 'A4W3W3', 'Q2LXP6', 'Q9WZ21', 'A5IJM4', 'B1L8U4', 'G7V5Y5', 'A4TRE8', 'A9R8F3', 'B2JZH0']
There are 25 DHADs to analyze.


------------ The next section is to download and create FASTA files from the ready DHAD uniprot IDs ----------------

In [4]:
#os.chdir(home_path)
# change path to fastafiles
#os.makedirs('fastafiles', exist_ok=True)
#os.chdir(home_path+'/fastafiles')
#print(f'path changed to {os.getcwd()}')
os.chdir('/Users/jeremydcortez/Bioconda_Playground/25-0219/003/')
selected_df = pd.read_csv('selectedDHADIDs.csv') #change file!
selected_candidates = selected_df['gene'].tolist()

# writing uniprot lists into a fasta file with organism name as header
counter = 0
fastafile = 'ID_selected0220.fasta'
with open(fastafile, "w") as output_handle:
    for v in selected_candidates:
        #print(v)
        #print(get_species_fasta_from_uniprot_id(v))
        species = get_species_fasta_from_uniprot_id(v)
        #print(species[0])
        output_handle.write('>'+species[0]+'\n'+species[1]+ '\n')
        counter += 1
    print(end= "\n")

print(f'Wrote {counter} sequences to fasta file.')


Wrote 19 sequences to fasta file.


In [84]:
# this cell is to save all the notebook outputs to a file

import nbformat
from nbconvert import PythonExporter

os.chdir(home_path)
# change path to fastafiles
os.makedirs('outputlogs', exist_ok=True)
os.chdir(home_path+'/outputlogs')
print(f'path changed to {os.getcwd()}')

def extract_outputs_to_text(notebook_path, output_file_path):
    """
    Extracts cell outputs from a Jupyter Notebook and saves them to a text file.

    Args:
        notebook_path (str): Path to the Jupyter Notebook file (.ipynb).
        output_file_path (str): Path to the output text file.
    """
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)

    outputs = []
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            for output in cell['outputs']:
                if output['output_type'] == 'stream':
                    outputs.append(output['text'])
                elif output['output_type'] == 'execute_result':
                    outputs.append(str(output['data'].get('text/plain', '')))
                elif output['output_type'] == 'display_data':
                     outputs.append(str(output['data'].get('text/plain', '')))
                elif output['output_type'] == 'error':
                    outputs.append('\n'.join(output['traceback']))

    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(outputs))

# Example usage:
notebook_path = home_path+'/GNNsSearch0216.ipynb'
output_file_path = 'notebook_outputsGNN'+str(N)+'.txt'
extract_outputs_to_text(notebook_path, output_file_path)

path changed to /Users/jeremydcortez/Bioconda_Playground/25-0214/001/outputlogs
